# ZeroBus + OpenTelemetry → Unity Catalog (Databricks OTLP)

This notebook sends **OTLP http/protobuf** to your **Databricks workspace** (`/api/2.0/otel/v1/...`) with **`X-Databricks-UC-Table-Name`** so traces and metrics land in **Unity Catalog Delta** tables. It still uses **ZeroBus ingest** for business rows (same pattern as `grafana/zerobus-otel.ipynb`).

**Prerequisites:** Create the three UC tables and grants per **`docs/zerobus-ingest-otel-uc.md`** (PrD/PrPr DDL). Set **`DATABRICKS_TOKEN`** (PAT or SP) with **SELECT** + **MODIFY** on those tables.

**Secrets (optional):** JSON in Databricks scope/key (defaults below) merged by **`merge_uc_otel_secret`**. Do not commit tokens.

**Kernel:** Do not mix with `grafana/zerobus-otel.ipynb` in the same session without restart (global OpenTelemetry providers).


In [ ]:
%pip install --quiet databricks-zerobus-ingest-sdk
%pip install --quiet opentelemetry-api opentelemetry-sdk opentelemetry-instrumentation
%pip install --quiet opentelemetry-exporter-otlp
%pip install --quiet opentelemetry-instrumentation-grpc opentelemetry-instrumentation-requests

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
_grafana_dir = None
_uc_dir = None
for _root in [_cwd, *_cwd.parents]:
    _g = _root / "notebooks" / "otel" / "grafana" / "zerobus_otel_lab.py"
    _u = _root / "notebooks" / "otel" / "uc" / "zerobus_uc_otel_lab.py"
    if _g.is_file() and _u.is_file():
        _grafana_dir = _g.parent
        _uc_dir = _u.parent
        break
if _grafana_dir is None or _uc_dir is None:
    raise RuntimeError("Could not find notebooks/otel/grafana and notebooks/otel/uc lab modules.")
sys.path.insert(0, str(_grafana_dir))
sys.path.insert(0, str(_uc_dir))

from zerobus_otel_lab import bootstrap_zerobus_with_otel, default_zerobus_config
from zerobus_uc_otel_lab import (
    create_databricks_uc_otel_telemetry,
    default_uc_otel_config,
    flush_databricks_uc_otel,
    merge_uc_otel_secret,
)

_UC_OTEL_SECRET_SCOPE = "lfczerobusdemo"
_UC_OTEL_SECRET_KEY = "uc_otel_config"
_ZEROBUS_SECRET_SCOPE = "lfczerobusdemo"
_ZEROBUS_SECRET_KEY = "lfczerobusdemo"

UC_OTEL_CONFIG = default_uc_otel_config(
    uc_otel_secret_scope=_UC_OTEL_SECRET_SCOPE,
    uc_otel_secret_key=_UC_OTEL_SECRET_KEY,
)
# Required for OTLP: workspace origin + PAT/SP token + UC prefix (tables must exist).
# UC_OTEL_CONFIG["DATABRICKS_WORKSPACE_URL"] = "https://<workspace>.cloud.databricks.com"
# UC_OTEL_CONFIG["DATABRICKS_TOKEN"] = dbutils.secrets.get(...)  # or PAT string
# UC_OTEL_CONFIG["UC_OTEL_CATALOG"] = "main"
# UC_OTEL_CONFIG["UC_OTEL_SCHEMA"] = ""  # empty → current_user() sanitized
# UC_OTEL_CONFIG["UC_OTEL_PREFIX"] = "zerobus_uc_demo"

merge_uc_otel_secret(UC_OTEL_CONFIG)

_config = default_zerobus_config(
    zerobus_secret_scope=_ZEROBUS_SECRET_SCOPE,
    zerobus_secret_key=_ZEROBUS_SECRET_KEY,
)

otel = create_databricks_uc_otel_telemetry(UC_OTEL_CONFIG, _config, spark)
ctx = bootstrap_zerobus_with_otel(otel, _config, spark)

trace_operation = otel.trace_operation
record_metrics = otel.record_metrics
_config = ctx.config
fq_table_name = ctx.fq_table_name


def _save_config_if_changed() -> None:
    ctx.save_config_if_changed()

In [ ]:
# INSTRUMENTED ZEROBUS INGESTION (same shape as grafana/zerobus-otel.ipynb)

import json
import time

from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties
from zerobus.sdk.sync import ZerobusSdk

with trace_operation(
    "zerobus_ingestion",
    {
        "table": fq_table_name,
        "endpoint": _config["ZEROBUS_SERVER_ENDPOINT"],
        "record_count": 10,
    },
) as main_span:
    with trace_operation("sdk_initialization"):
        sdk = ZerobusSdk(_config["ZEROBUS_SERVER_ENDPOINT"], _config["DATABRICKS_WORKSPACE_URL"])

    with trace_operation(
        "stream_creation", {"table": fq_table_name, "record_type": "JSON"}
    ) as stream_span:
        table_properties = TableProperties(fq_table_name)
        options = StreamConfigurationOptions(record_type=RecordType.JSON)
        stream = sdk.create_stream(
            _config["ZEROBUS_APP_ID"],
            _config["ZEROBUS_OAUTH_SECRET"],
            table_properties,
            options,
        )
        stream_span.set_attribute("stream_created", True)

    try:
        last_offset = None
        total_bytes = 0
        batch_start = time.time()

        for i in range(10):
            with trace_operation(
                f"ingest_record_{i}", {"record_index": i, "device_name": f"sensor-{i}"}
            ) as record_span:
                record_dict = {
                    "device_name": f"sensor-{i}",
                    "temp": 20 + i % 15,
                    "humidity": 50 + i % 40,
                }
                record_bytes = len(json.dumps(record_dict).encode("utf-8"))
                total_bytes += record_bytes
                record_span.set_attribute("record_bytes", record_bytes)
                last_offset = stream.ingest_record_offset(record_dict)
                record_span.set_attribute("offset", str(last_offset) if last_offset else "none")

        batch_duration_ms = (time.time() - batch_start) * 1000
        record_metrics("batch_ingest", records=10, bytes_size=total_bytes, duration_ms=batch_duration_ms)
        main_span.set_attribute("total_bytes", total_bytes)
        main_span.set_attribute("batch_duration_ms", batch_duration_ms)

        if last_offset is not None:
            with trace_operation("wait_for_commit", {"offset": str(last_offset)}) as wait_span:
                stream.wait_for_offset(last_offset)
                wait_span.set_attribute("wait_duration_ms", (time.time() - batch_start) * 1000)

    finally:
        with trace_operation("stream_close"):
            stream.close()

print("✅ Ingestion complete. Query UC OTEL span/metric tables (see flush cell).")

In [ ]:
# Flush OTLP to UC + save Zerobus config if changed

flush_databricks_uc_otel(ctx)